# State machine on real gas data

Gas chain output drives the box state machine (spec figure 3). Question: how often does clean air wake the camera, and how fast do real burns reach Suspicion?

- **Gas only:** no flame and no camera result are injected. Gas alone can reach Suspicion, never Fire (tested).
- **Sensor:** BME688/690 data, cadence about 25 to 111 s. The box scans every 30 s in Alert mode, so the data is used as Alert-mode scans.
- **Log:** RES-SM-1, RES-SM-2 in `docs/research/box-state-machine.md`.

In [1]:
import sys
import numpy as np
sys.path.insert(0, "../src"); sys.path.insert(0, "../../../gas_chain/python/src")
from gas_chain import GasChain, SMOKE, FAULT
from gas_chain.data_mendeley import load_all
from state_machine import StateMachine, Inputs, NORMAL, SUSPICION, FIRE

## Method

- One gas chain and one state machine per episode, same default settings as the unit tests.
- FWI is forced to 'high' (mode Alert, pump rule active) so the worst case for wakes and pump is measured.
- Clean air = phase 0 after warm-up. Strong smoke = phases 3 to 5.

In [2]:
D = load_all()

def replay_episode(ep):
    """Gas chain -> state machine, scan by scan. No flame, no camera result: gas can only reach Suspicion."""
    idx = np.flatnonzero(D["episode"] == ep)
    chain, sm = GasChain(), StateMachine()
    rows = []
    for i in idx:
        r = chain.update(D["gas"][i], D["temp"][i], D["rh"][i], D["dt"][i])
        out = sm.step(Inputs(gas_smoke=r.cls == SMOKE, gas_fault=r.cls == FAULT, fwi_mode="alert",
                             fwi_danger=3, dt_s=float(D["dt"][i])))
        rows.append((D["phase"][i], D["dt"][i], out.state, out.camera_on, r.warmup, out.actuator))
    return rows

episodes = {ep: replay_episode(ep) for ep in np.unique(D["episode"])}
print(len(episodes), "episodes replayed")

38 episodes replayed


In [3]:
clean_s = wakes = susp_clean = entries = 0
actuator_clean = 0
for ep, rows in episodes.items():
    prev = NORMAL
    for ph, dt, state, cam, warm, act in rows:
        if warm or ph != 0:
            prev = state
            continue
        clean_s += dt
        wakes += cam
        susp_clean += dt if state != NORMAL else 0
        entries += (state == SUSPICION and prev == NORMAL)
        actuator_clean += dt if act else 0
        prev = state
h = clean_s / 3600
print(f"clean air observed: {h:.2f} h in {len(episodes)} episodes")
print(f"  entries into Suspicion: {entries}   ({entries / h:.1f} per hour)")
print(f"  camera wakes:           {wakes}   ({wakes / h:.1f} per hour)")
print(f"  time spent in Suspicion: {susp_clean / clean_s * 100:.1f} % of clean time")
print(f"  pump on (FWI forced to 'high' here): {actuator_clean / clean_s * 100:.1f} % of clean time")

clean air observed: 13.13 h in 38 episodes
  entries into Suspicion: 4   (0.3 per hour)
  camera wakes:           28   (2.1 per hour)
  time spent in Suspicion: 5.3 % of clean time
  pump on (FWI forced to 'high' here): 5.3 % of clean time


In [4]:
print("Episodes that enter Suspicion while the air is clean:")
for ep, rows in episodes.items():
    c = [(s, w) for ph, dt, s, cam, w, a in rows if ph == 0 and not w]
    n_susp = sum(1 for s, w in c if s != NORMAL)
    if n_susp:
        i = np.flatnonzero(D["episode"] == ep)[0]
        print(f"  {D['kind'][ep]:<22} burn {D['experiment'][i]}  clean scans {len(c):>3}  in Suspicion {n_susp:>3}")

Episodes that enter Suspicion while the air is clean:
  node3_profile2_s18     burn 2025  clean scans  15  in Suspicion  10
  node3_profile2_s19     burn 2025  clean scans  15  in Suspicion   8
  node3_profile3_s20     burn 2025  clean scans   7  in Suspicion   5
  node3_profile3_s21     burn 2025  clean scans   7  in Suspicion   5


**Reading.**

- All 4 entries into Suspicion on clean air come from the same 4 episodes as the false smoke of RES-GAS-1 (node 3, heater profiles 2 and 3, slow scans). The other 34 episodes: none.
- Averaged over all clean time: 0.3 entries and 2.1 camera wakes per hour, 5.3 % of the time in Suspicion. The average hides the concentration: in a node affected by power-on drift, the camera wakes every 30 s for as long as the drift is read as smoke.
- Nothing in the machine lets a confident `background` camera result calm the wake rate. Gas has to clear for 10 scans. **Open:** back-off of the camera period after a confident `background`.

In [5]:
delays, never = [], []
for ep, rows in episodes.items():
    t = 0.0; start2 = None; first = None
    for ph, dt, state, cam, warm, act in rows:
        t += dt
        if ph >= 2 and start2 is None:
            start2 = t
        if start2 is not None and first is None and state != NORMAL and not warm:
            first = t
    if start2 is None:
        continue
    (delays if first is not None else never).append((first - start2) / 60 if first is not None else None)
d = np.array([x for x in delays])
print(f"episodes with a burn: {len(delays) + len(never)}   reached Suspicion: {len(delays)}   never: {len(never)}")
print(f"minutes from the start of phase 2 to Suspicion: median {np.median(d):.1f}, 90th percentile {np.percentile(d, 90):.1f}, max {d.max():.1f}")

episodes with a burn: 38   reached Suspicion: 38   never: 0
minutes from the start of phase 2 to Suspicion: median 3.1, 90th percentile 7.5, max 7.9


In [6]:
susp_burn_s = wakes_burn = 0
for ep, rows in episodes.items():
    for ph, dt, state, cam, warm, act in rows:
        if ph in (3, 4, 5) and not warm:
            susp_burn_s += dt
            wakes_burn += cam
print(f"strong smoke observed: {susp_burn_s / 3600:.2f} h -> camera wakes {wakes_burn} ({wakes_burn / (susp_burn_s / 3600):.0f} per hour)")

strong smoke observed: 37.37 h -> camera wakes 3111 (83 per hour)


**Reading.**

- All 38 episodes reach Suspicion, median 3.1 min after the start of phase 2 (90th percentile 7.5, max 7.9). This is the delay of the gas chain: the machine adds none, because the chain already debounces the class.
- In strong smoke the camera wakes 83 times per hour. Energy per capture is not known (spec 3.2: to measure on the board). If a capture is costly, a longer period once a first result is in is needed.
- These replays have no camera result and no flame, so none of them reaches Fire. Fire paths are covered by the unit tests only.

## Decision

- 18 unit tests pass: every rule of the fiche has one, plus "gas alone or flame alone never reaches Fire".
- Thresholds are provisional assumptions: 2 flame reads, 10 clear scans, 30 min Fire hold, camera confidence 0.6, camera period 30 s.
- **Not shown by this notebook:** Fire on real data (no camera block yet, no real flame data), Eco cadence (300 s), neighbour boxes, energy.
- **Next:** camera CNN block (96x96, fire / smoke / background) to produce the `camera` input; then the C/C++ port of the three box blocks with shared test vectors.